# Capítulo 4 — Razonar con probabilidades · Solución comentada

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch04_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y luego muestra su equivalente en Python. En cada cálculo, los comentarios indican el grupo que sirve de denominador.

**Índice.** Datos · Los cálculos de las lecciones 4.1 a 4.5 · Ejercicios 4.1 a 4.6

**Antes de cada cálculo**, pregúntate en qué conjunto estás contando: spams, mensajes marcados o mensajes recibidos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLUE, ORANGE = '#2a78d6', '#eb6834'   # «ningún spam se escapa», curva de la distribución normal
from math import comb

## Datos

Los identificadores D4.1 y D4.2 son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 4, copiados de los archivos CSV del libro (no modificar)
# Filas: spam, legítimo. Columnas: marcado, no marcado.
ROWS, COLUMNS = ['spam', 'legítimo'], ['marcado', 'no marcado']

# D4.1 — buzón donde el 20 % de los mensajes son spam (1 000 mensajes)
inbox_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=ROWS, columns=COLUMNS)

# D4.2 — mismo filtro, buzón donde el 2 % de los mensajes son spam (1 000 mensajes)
inbox_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=ROWS, columns=COLUMNS)

# D4.3 — 1 000 tiempos de procesamiento (ms), regenerados idénticos al archivo CSV
durations = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `inbox_20` | D4.1: buzón donde el 20 % de los mensajes son spam (filas spam / legítimo, columnas marcado / no marcado) | mensajes | — |
| `inbox_2` | D4.2: mismo filtro, buzón donde el 2 % de los mensajes son spam | mensajes | — |
| `durations` | D4.3: 1000 tiempos de procesamiento | ms | — |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 4.1 a 4.5.

In [ ]:
# Lección 4.1 — contar y luego dividir por el grupo correcto (ecuación 4.1)
t = inbox_20
n = t.values.sum()
print('total:', n, '| spams:', t.loc['spam'].sum(), '| marcados:', t['marcado'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(marcado) =', t['marcado'].sum() / n,
      '| P(spam y marcado) =', t.loc['spam', 'marcado'] / n)
# Entre los marcados: el denominador es la columna «marcado»
print(f"spams entre los marcados: {t.loc['spam', 'marcado']} / {t['marcado'].sum()} "
      f"≈ {t.loc['spam', 'marcado'] / t['marcado'].sum():.2f}")

In [ ]:
# Lección 4.2 — el mismo filtro en un buzón donde los spams son raros (ecuación 4.2)
d, f = 0.90, 0.05                   # proporciones marcadas: de los spams, de los mensajes legítimos
for p, t in [(0.20, inbox_20), (0.02, inbox_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    rebuilt = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(rebuilt) == t.values).all()   # la tabla se deduce de las tasas
    print(f'tasa base {p:.0%}: {t.values.tolist()}  →  proporción de spams entre los marcados '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lección 4.3 — dos probabilidades condicionadas, dos denominadores (ecuación 4.3)
def conditional(t, case, group):
    """Proporción de la casilla (fila, columna) dentro del grupo: una fila ('spam') o una columna ('marcado')."""
    denominator = t.loc[group].sum() if group in t.index else t[group].sum()
    return t.loc[case] / denominator, denominator

for name, t in [('20 %', inbox_20), ('2 %', inbox_2)]:
    recall, den_r = conditional(t, ('spam', 'marcado'), 'spam')
    precision, den_p = conditional(t, ('spam', 'marcado'), 'marcado')
    print(f'buzón con {name}: P(marcado | spam) = {recall:.2f} (denominador {den_r}); '
          f'P(spam | marcado) = {precision:.2f} (denominador {den_p})')

# Los mismos números con scikit-learn, a partir de etiquetas mensaje por mensaje (si está instalado)
try:
    from sklearn.metrics import precision_score, recall_score
    truth = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predicted_label = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = marcado
    print(f'scikit-learn: exhaustividad = {recall_score(truth, predicted_label):.2f}, precisión = {precision_score(truth, predicted_label):.2f}')
except ImportError:
    print('scikit-learn no está instalado: este cálculo es opcional.')

In [ ]:
# Lección 4.4 — el teorema de Bayes (ecuaciones 4.4 y 4.5) y la curva del libro
def bayes(p, d=0.90, f=0.05):
    p_flagged = d * p + f * (1 - p)                  # ecuación 4.5
    return d * p / p_flagged                        # ecuación 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p}: P(marcado) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | marcado) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=FLAGGED, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Tasa base: proporción de spams (%)'); plt.ylabel('P(spam | marcado)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lección 4.5 — diez spams: distribución binomial (ecuación 4.7) y suceso complementario (ecuación 4.8)
n, p = 10, 0.1                        # contamos sobre 10 mensajes SPAM; p = probabilidad de que se escape
distribution = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| al menos uno se escapa =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 a 4:', [round(v, 3) for v in distribution[:5]], '| suma =', round(sum(distribution), 6))
colors = [BLUE] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), distribution, color=colors, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Número de spams que se escapan entre 10 spams'); plt.ylabel('Probabilidad'); plt.show()

# Un vistazo a la distribución normal: tiempos de procesamiento (D4.3)
m, s = durations.mean(), durations.std()
print(f'media = {m:.2f} ms, desviación típica = {s:.2f} ms; a ±1 desviación típica: {np.mean(abs(durations - m) <= s):.1%}, '
      f'a ±2 desviaciones típicas: {np.mean(abs(durations - m) <= 2 * s):.1%}')

## Ejercicio 4.1 — Leer un gráfico: veinte spams

*Nociones: distribución binomial, suceso complementario (lección 4.5).* Primero lee la figura del libro.

### a) a d) Solución comentada

Aquí contamos sobre **20 spams**, no sobre mensajes recibidos.

**a)** Ningún spam se escapa: alrededor de 0,36. Exactamente uno: alrededor de 0,38.

**b)** La barra más alta es la de 1: que se escape un spam es el caso más probable.

**c)** $1 - 0{,}36 \approx 0{,}64$. Con la ecuación 4.8: $1 - 0{,}95^{20} \approx 0{,}6415$.

**d)** Cada spam tiene la mitad de probabilidades de escaparse (5 % en lugar de 10 %), pero hay el doble de spams (20 en lugar de 10). En promedio, el filtro sigue dejando escapar uno: $20 \times 0{,}05 = 1$. Por eso la probabilidad de que se escape al menos uno sigue siendo parecida: 64 % frente a 65 %.

In [ ]:
def binomial(n, p, k):
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

In [ ]:
# Comprobación con los 20 spams del ejercicio
if binomial(20, 0.05, 0) is None:
    print('Completa la función.')
else:
    p0, p1 = binomial(20, 0.05, 0), binomial(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, al menos uno = {1 - p0:.4f} →',
          'correcto' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'revisar')

## Ejercicio 4.2 — Un filtro más prudente

*Nociones: probabilidad condicionada (lección 4.3), teorema de Bayes (lección 4.4), tasa base (lección 4.2).* El filtro prudente marca el 75 % de los spams y el 1 % de los mensajes legítimos. Buzón de 10 000 mensajes, de los cuales el 10 % son spams. Haz primero las preguntas a) a e) en papel.

### a) a c) Solución comentada

**a)** Empieza por las filas, es decir, por lo que son los mensajes. El 10 % de 10 000 da **1 000 spams**, y quedan **9 000 mensajes legítimos**. El filtro marca el 75 % de los spams: 0,75 × 1 000 = **750**, así que se escapan 250 spams. Marca el 1 % de los mensajes legítimos: 0,01 × 9 000 = **90** falsas alarmas, y pasan 8 910 mensajes legítimos.

| | Marcado | No marcado | Total |
|:--|--:|--:|--:|
| Spam | 750 | 250 | 1 000 |
| Legítimo | 90 | 8 910 | 9 000 |
| **Total** | **840** | **9 160** | **10 000** |

**b)** $P(\text{marcado} \mid \text{spam})$: contamos **en la fila de los spams**, que sirve de denominador. $750 / 1\,000 = 0{,}75$. Recuperamos la tasa anunciada, lo cual tranquiliza. $P(\text{spam} \mid \text{marcado})$: contamos **en la columna de los marcados**. $750 / 840 \approx 0{,}893$. Mismo numerador, 750; denominadores distintos, 1 000 y 840.

**c)** Con la ecuación 4.5: $P(\text{marcado}) = 0{,}75 \times 0{,}1 + 0{,}01 \times 0{,}9 = 0{,}075 + 0{,}009 = 0{,}084$, es decir, los 840 mensajes marcados de 10 000. Con la ecuación 4.4: $P(\text{spam} \mid \text{marcado}) = 0{,}075 / 0{,}084 \approx 0{,}893$. Bayes no hace más que reescribir el recuento de la tabla con proporciones.

### a) La tabla en Python

In [ ]:
spam_flagged = round(0.75 * 1000)          # 750: el 75 % de los 1 000 spams
spam_not_flagged = 1000 - spam_flagged     # 250
legit_flagged = round(0.01 * 9000)      # 90 falsas alarmas: el 1 % de los 9 000 legítimos
legit_not_flagged = 9000 - legit_flagged   # 8 910

In [ ]:
# Comprobación
cells = [spam_flagged, spam_not_flagged, legit_flagged, legit_not_flagged]
if None in cells:
    print('Completa la celda anterior.')
else:
    cautious = pd.DataFrame([cells[:2], cells[2:]], index=ROWS, columns=COLUMNS)
    print(cautious)
    print('tabla:', 'correcto' if cells == [750, 250, 90, 8910] else 'revisar')

### b) Las dos probabilidades condicionadas

In [ ]:
p_flagged_if_spam = 750 / (750 + 250)   # denominador: la fila de los spams
p_spam_if_flagged = 750 / (750 + 90)    # denominador: la columna de los marcados

In [ ]:
# Comprobación
for name, yours, truth in [('P(marcado | spam)', p_flagged_if_spam, 0.75),
                            ('P(spam | marcado)', p_spam_if_flagged, 750 / 840)]:
    print(f'{name}:', 'aún sin completar' if yours is None else
          ('correcto' if abs(yours - truth) < 0.001 else 'revisar'))

### c) a e) El teorema de Bayes como función

In [ ]:
def flagged_spam_probability(p, d, f):
    p_flagged = d * p + f * (1 - p)     # ecuación 4.5: las dos casillas de la columna «marcado»
    return d * p / p_flagged            # ecuación 4.4: la casilla roja dividida por la columna

In [ ]:
# Comprobación de c), d) y e)
if flagged_spam_probability(0.1, 0.75, 0.01) is None:
    print('Completa la función.')
else:
    scenario = [('c) prudente, 10 % de spams', (0.10, 0.75, 0.01), 0.8929),
           ('d) prudente, 1 % de spams', (0.01, 0.75, 0.01), 0.4310),
           ('e) prudente, 2 % de spams', (0.02, 0.75, 0.01), 0.6048),
           ('e) filtro de las lecciones, 2 % de spams', (0.02, 0.90, 0.05), 0.2687)]
    for name, args, expected in scenario:
        v = flagged_spam_probability(*args)
        print(f'{name}: {v:.4f} →', 'correcto' if abs(v - expected) < 0.0001 else 'revisar')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, flagged_spam_probability(P, 0.90, 0.05), color=FLAGGED, linewidth=2,
             label='filtro de las lecciones (90 %; 5 %)')
    plt.plot(100 * P, flagged_spam_probability(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='filtro prudente (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Tasa base: proporción de spams (%)'); plt.ylabel('P(spam | marcado)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) y e) Solución comentada

**d)** Predicción esperada: la probabilidad **disminuye**, ya que los spams se vuelven más raros y las falsas alarmas pesan más (lección 4.2). Para 10 000 mensajes: 100 spams, de los cuales 75 marcados; 9 900 mensajes legítimos, de los cuales 99 marcados. En la columna de los marcados: $75 / (75 + 99) = 75 / 174 \approx 0{,}431$. Menos de una de cada dos marcas es un spam real, aunque el filtro no ha cambiado.

**e)** Para el filtro prudente en el buzón con 2 %: $\dfrac{0{,}75 \times 0{,}02}{0{,}75 \times 0{,}02 + 0{,}01 \times 0{,}98} = \dfrac{0{,}015}{0{,}0248} \approx 0{,}605$, frente a 0,27 para el filtro de las lecciones. Comparar estos dos números es comparar la **precisión** de los dos filtros: contamos en la columna de los mensajes marcados. Cuando el filtro prudente marca un mensaje, puedes fiarte más: da cinco veces menos falsas alarmas. Pero para elegir un filtro, la precisión no basta. Cuenta también en la fila de los spams: de 100 spams, el filtro prudente deja escapar 25 y el filtro de las lecciones 10 (su **exhaustividad** vale 0,90 frente a 0,75). La buena elección depende de lo que más cuesta: un mensaje importante perdido en «Correo no deseado» o un spam más en la bandeja de entrada. Como en el capítulo 3, hay que decidir la medida antes de comparar. La curva muestra que la diferencia entre los dos filtros es mayor cuando los spams son raros.

## Ejercicio 4.3 — Tres spams y luego diez mensajes

*Nociones: ensayo de Bernoulli y distribución binomial (lección 4.5), probabilidad condicionada (lección 4.3).* Haz primero las preguntas en papel.

### a) a e) Solución comentada

**a)** Cada rama multiplica 0,9 (marcado) o 0,1 (se escapa). El resultado «tres marcados» vale $0{,}9^3 = 0{,}729$; cada resultado con un solo spam que se escapa vale $0{,}1 \times 0{,}9^2 = 0{,}081$; con dos que se escapan, $0{,}1^2 \times 0{,}9 = 0{,}009$; los tres, $0{,}1^3 = 0{,}001$.

**b)** Tres resultados tienen un solo spam que se escapa: $P(X = 1) = 3 \times 0{,}081 = 0{,}243$. Del mismo modo $P(X = 2) = 3 \times 0{,}009 = 0{,}027$, $P(X = 0) = 0{,}729$, $P(X = 3) = 0{,}001$. Suma: $0{,}729 + 0{,}243 + 0{,}027 + 0{,}001 = 1$. El factor 3 es el coeficiente $\binom{3}{1} = \binom{3}{2} = 3$.

**c)** Sumando: $0{,}243 + 0{,}027 + 0{,}001 = 0{,}271$. Por el complementario: $1 - 0{,}729 = 0{,}271$. El segundo cálculo es más corto.

**d)** **Comprueba en qué conjunto estás contando.** Ya no contamos sobre diez *spams*, sino sobre diez *mensajes recibidos*, algunos de los cuales son legítimos. Para un mensaje recibido, «spam que se escapa» significa «spam **y** no marcado»: $0{,}2 \times 0{,}1 = 0{,}02$ (lección 4.3). Ninguno de los diez se escapa con probabilidad $0{,}98^{10} \approx 0{,}817$, así que al menos un spam se escapa con probabilidad $1 - 0{,}98^{10} \approx 0{,}183$.

**e)** En la lección 4.5, los diez mensajes son todos spams: cada uno tiene un 10 % de probabilidad de escaparse. Entre diez mensajes recibidos solo hay, en promedio, dos spams, y cada mensaje tiene solo un 2 % de probabilidad de ser un spam que se escapa. «Diez spams» y «diez mensajes recibidos» son dos preguntas distintas.

In [ ]:
distribution_3 = [comb(3, k) * 0.1 ** k * 0.9 ** (3 - k) for k in range(4)]   # sobre 3 mensajes SPAM
p_d = 1 - (1 - 0.2 * 0.1) ** 10   # sobre 10 MENSAJES RECIBIDOS: spam Y no marcado = 0,02

In [ ]:
# Comprobación
if distribution_3 is None or p_d is None:
    print('Completa la celda anterior.')
else:
    print('b):', [round(v, 3) for v in distribution_3], '→', 'correcto' if np.allclose(distribution_3, [0.729, 0.243, 0.027, 0.001]) else 'revisar')
    print(f'd): {p_d:.4f} →', 'correcto' if abs(p_d - 0.1829) < 0.0001 else 'revisar')

## Ejercicio 4.4 — Simular diez spams

*Nociones: distribución binomial (lección 4.5), frecuencia y probabilidad.* Contamos sobre **diez spams**.

In [ ]:
def series_simulation(rng):
    """Devuelve el número de spams que se escapan entre 10. rng.random() saca un número entre 0 y 1."""
    return int(np.sum(rng.random(10) < 0.1))   # un sorteo por spam: se escapa si < 0,1

In [ ]:
# b) y c) 10 000 series (semilla fija para obtener los mismos números)
rng = np.random.default_rng(4)
if series_simulation(rng) is None:
    print('Completa la función.')
else:
    rng = np.random.default_rng(4)
    X = np.array([series_simulation(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'frecuencia observada: {freq:.4f}; probabilidad calculada: {1 - 0.9 ** 10:.4f} →',
          'correcto' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'revisar')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=FLAGGED, label='frecuencia observada')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probabilidad (ecuación 4.7)')
    plt.xlabel('Spams que se escapan entre 10'); plt.legend(); plt.show()
    # d) Más series: la frecuencia se acerca a la probabilidad
    for N in (100, 100, 100, 100000):
        Y = np.array([series_simulation(rng) for _ in range(N)])
        print(f'{N:>6} series: frecuencia = {np.mean(Y >= 1):.4f}')

### Solución comentada

Con la semilla del notebook, 10 000 series dan una frecuencia de 0,6549, para una probabilidad de 0,6513. Las dos son cercanas, pero no iguales: una frecuencia observada varía de una simulación a otra. Con 100 series, puede alejarse varios puntos porcentuales (vuelve a ejecutar la celda sin semilla para verlo). Con 100 000 series, se acerca a unas pocas milésimas. La probabilidad es el valor hacia el que tiende la frecuencia cuando aumenta el número de series. Comprueba también el conjunto: cada serie cuenta **spams**, nunca mensajes recibidos.

## Ejercicio 4.5 — Predecir y luego comprobar: una campaña de spam

*Nociones: independencia, suceso complementario (lección 4.5).*

### a) a c) y e) Solución comentada

**b)** El filtro deja escapar los diez spams de golpe con probabilidad 0,1, o no deja escapar ninguno. El número de spams que se escapan vale, por tanto, 0 o 10. Al menos uno se escapa con probabilidad **0,1**, mucho menos que el 65 %.

**c)** En promedio: $10 \times 0{,}1 + 0 \times 0{,}9 = 1$ spam que se escapa. Es la misma media que en la lección 4.5 ($10 \times 0{,}1 = 1$). La media es idéntica, pero la distribución es muy distinta.

**e)** La independencia ya no se cumple: saber que el primer spam se escapa hace que los demás se escapen casi con seguridad. La probabilidad de escaparse, 0,1 para cada uno, no ha cambiado. Para el usuario, diez spams de golpe son más molestos, y más visibles. La distribución binomial solo se aplica con sus dos hipótesis.

In [ ]:
def campaign_simulation(rng):
    return 10 if rng.random() < 0.1 else 0   # un único sorteo para toda la campaña

In [ ]:
# d) Comprobación por simulación
rng = np.random.default_rng(5)
if campaign_simulation(rng) is None:
    print('Completa la función.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([campaign_simulation(rng) for _ in range(10000)])
    print(f'al menos uno se escapa: {np.mean(Z >= 1):.4f}; media: {Z.mean():.3f} →',
          'correcto' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'revisar')

## Ejercicio 4.6 — Desafío: ¿cuántos spams hacen falta para superar un 99 % de riesgo de que se escape uno?

*Nociones: distribuciones de Bernoulli y binomial, distribución normal (lección 4.5), desviación típica (lección 1.1).*

### a) a d) Solución comentada

**a)** Sea $X_i = 1$ si el spam $i$ se escapa, 0 en caso contrario. Entonces $X = X_1 + \dots + X_n$: sumar los 1 cuenta los spams que se escapan. Cada $X_i$ vale en media $1 \times p + 0 \times (1 - p) = p$, y la media de una suma es la suma de las medias: $n\,p$. Para 10 spams, 1.

**b)** Hace falta $1 - 0{,}9^n > 0{,}99$, es decir, $0{,}9^n < 0{,}01$. Se obtiene $0{,}9^{43} \approx 0{,}0108$ y $0{,}9^{44} \approx 0{,}0097$: hacen falta **44 spams**.

**c)** Media: $100 \times 0{,}1 = 10$. Desviación típica: $\sqrt{100 \times 0{,}1 \times 0{,}9} = \sqrt{9} = 3$.

**d)** Las barras de la distribución binomial casi se ajustan a la curva en campana de media 10 y desviación típica 3. Cuando $n$ es grande, la distribución binomial se parece a una distribución normal: por eso la curva de la lección 4.5 aparecerá a menudo.

In [ ]:
n_min = next(n for n in range(1, 200) if 0.9 ** n < 0.01)   # probamos n = 1, 2, 3…
mean_100 = 100 * 0.1
std_100 = np.sqrt(100 * 0.1 * 0.9)

In [ ]:
# Comprobación y luego d) binomial frente a normal
if None in (n_min, mean_100, std_100):
    print('Completa la celda anterior.')
else:
    print(f'n_min = {n_min} →', 'correcto' if n_min == 44 else 'revisar')
    print(f'media = {mean_100:.1f}, desviación típica = {std_100:.1f} →',
          'correcto' if np.isclose(mean_100, 10) and np.isclose(std_100, 3) else 'revisar')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='distribución binomial (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='distribución normal (media 10, desviación típica 3)')
    plt.xlabel('Spams que se escapan entre 100'); plt.legend(); plt.show()